
## DDL Gold: pf.gold.dim_fecha  (Type 0 - calendario estatico)
## Serializable, no SCD. Se construye una vez y se reutiliza.

In [0]:
%sql

DROP TABLE IF EXISTS pf.gold.dim_fecha;

CREATE TABLE IF NOT EXISTS pf.gold.dim_fecha (
    fecha_id BIGINT NOT NULL COMMENT 'PK - AAAAMMDD',
    fecha DATE NOT NULL COMMENT 'Fecha calendario',
    anio INT NOT NULL COMMENT 'Anio',
    trimestre INT NOT NULL COMMENT 'Trimestre 1-4',
    mes INT NOT NULL COMMENT 'Mes 1-12',
    nombre_mes STRING NOT NULL COMMENT 'Nombre del mes (es)',
    dia INT NOT NULL COMMENT 'Dia del mes',
    dia_semana STRING NOT NULL COMMENT 'Nombre del dia (es)',
    es_fin_de_semana BOOLEAN  NOT NULL COMMENT 'True si sabado/domingo',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (fecha_id)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Tiempo - Type 0 (calendario estatico 2020-2032)';

In [0]:
%sql

-- Poblado idempotente (MERGE) con calendario 2020-2032
CREATE OR REPLACE TEMP VIEW v_dates AS
SELECT
    CAST(DATE_FORMAT(d, 'yyyyMMdd') AS BIGINT) AS fecha_id,
    d AS fecha,
    YEAR(d) AS anio,
    QUARTER(d) AS trimestre,
    MONTH(d) AS mes,
    INITCAP(DATE_FORMAT(d, 'MMMM')) AS nombre_mes,
    DAYOFMONTH(d) AS dia,
    DATE_FORMAT(d, 'EEEE') AS dia_semana,
    (DAYOFWEEK(d) IN (1, 7)) AS es_fin_de_semana
FROM (
    SELECT EXPLODE(SEQUENCE(
        TO_DATE('2020-01-01'), TO_DATE('2032-12-31'), INTERVAL 1 DAY
    )) AS d
);

MERGE INTO pf.gold.dim_fecha AS target
USING v_dates AS src
ON target.fecha_id = src.fecha_id
WHEN NOT MATCHED THEN
  INSERT (fecha_id, fecha, anio, trimestre, mes, nombre_mes, dia, dia_semana, es_fin_de_semana, _createdAt)
  VALUES (src.fecha_id, src.fecha, src.anio, src.trimestre, src.mes, src.nombre_mes,
          src.dia, src.dia_semana, src.es_fin_de_semana, CURRENT_TIMESTAMP());

In [0]:
%sql

SELECT COUNT(*) AS total_dias FROM pf.gold.dim_fecha;